In [1]:
# Clone your GitHub repo (you’ll be prompted to authorize if it's private)
!git clone https://github.com/colterwood/LHL-final-final-project.git

Cloning into 'LHL-final-final-project'...
remote: Enumerating objects: 320, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 320 (delta 37), reused 55 (delta 14), pack-reused 224 (from 1)
Receiving objects: 100% (320/320), 5.39 MiB | 4.76 MiB/s, done.
Resolving deltas: 100% (152/152), done.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, ParameterGrid
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import files
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
import json
import os

In [3]:
# load the player game logs CSV from the data folder
df = pd.read_csv("LHL-final-final-project/data/2024_merged_gamelogs.csv")


# preview
df.head()

,team,g_num,month,day,home_away,opp,win_loss,team_score,opp_score,team_fg,...,day_of_week_by_team_travel_distance,team_vs_opp_median_score_by_team_travel_distance,team_vs_opp_homeaway_median_score_by_team_travel_distance,team_home_or_away_median_score_by_team_travel_distance,team_home_or_away_median_allowed_by_team_travel_distance,team_day_median_score_by_team_travel_distance,team_day_median_allowed_by_team_travel_distance,travel_distance_by_team_travel_distance,median_score_for_by_team_travel_distance,median_score_against_by_team_travel_distance
0,ATL,1,5,15,2,LAS,1,92,81,34,...,3.0,75.0,70.0,78.5,79.5,76.0,80.0,0.0,77.0,79.5
1,ATL,2,5,18,2,PHO,2,85,88,27,...,2.0,76.0,76.0,78.0,80.5,76.0,80.0,1.0,77.5,77.5
2,ATL,3,5,21,1,DAL,1,83,78,30,...,4.0,81.0,75.5,78.0,80.5,73.0,78.0,3.0,81.0,85.0
3,ATL,4,5,26,1,MIN,2,79,92,31,...,3.0,75.0,70.0,78.5,79.5,76.0,80.0,0.0,77.0,79.5
4,ATL,5,5,29,2,WAS,1,73,67,26,...,4.0,75.0,76.5,78.0,80.5,76.0,80.0,2.0,78.0,80.0


In [4]:
for col in df.columns:
  print(col)

team
g_num
month
day
home_away
opp
win_loss
team_score
opp_score
team_fg
team_fga
team_fg_pct
team_3p
team_3pa
team_3p_pct
team_ft
team_fta
team_ft_pct
team_orb
team_trb
team_ast
team_stl
team_blk
team_tov
team_pf
opponent_fg
opponent_fga
opponent_fg_pct
opponent_3p
opponent_3pa
opponent_3p_pct
opponent_ft
opponent_fta
opponent_ft_pct
opponent_orb
opponent_trb
opponent_ast
opponent_stl
opponent_blk
opponent_tov
opponent_pf
advanced_ortg
advanced_drtg
advanced_pace
advanced_ftr
advanced_3par
advanced_ts_pct
advanced_trb_pct
advanced_ast_pct
advanced_stl_pct
advanced_blk_pct
offensive_four_factors_efg_pct
offensive_four_factors_tov_pct
offensive_four_factors_orb_pct
offensive_four_factors_ft_per_fga
defensive_four_factors_efg_pct
defensive_four_factors_tov_pct
defensive_four_factors_drb_pct
defensive_four_factors_ft_per_fga
day_of_week
team_vs_opp_median_score
team_vs_opp_homeaway_median_score
team_home_or_away_median_score
team_home_or_away_median_allowed
team_day_median_score
team_day_

In [5]:
# load the player game logs CSV from the data folder
player_df = pd.read_csv("LHL-final-final-project/data/player_data.csv")


# preview
player_df.head()

,player,year,tm,age,g,gs,per_game_mp,per_game_fg,per_game_fga,per_game_fg_pct,...,pbp_plus_minus_per_100_poss_on_off,pbp_turnovers_badpass,pbp_turnovers_lostball,pbp_fouls_committed_shoot,pbp_fouls_committed_off,pbp_fouls_drawn_shoot,pbp_fouls_drawn_off,pbp_misc_pga,pbp_misc_and1,pbp_misc_blkd
0,Diana Taurasi,2004,PHO,22.0,34.0,34.0,33.2,6.1,14.8,0.416,...,1.6,46.0,13.0,38.0,27.0,0.0,0.0,317.0,18.0,18.0
1,Diana Taurasi,2005,PHO,23.0,33.0,33.0,33.0,5.3,12.9,0.410,...,4.8,61.0,20.0,42.0,22.0,0.0,0.0,342.0,8.0,12.0
2,Diana Taurasi,2006,PHO,24.0,34.0,34.0,33.9,8.8,19.4,0.452,...,14.0,33.0,16.0,53.0,18.0,71.0,2.0,299.0,17.0,19.0
3,Diana Taurasi,2007,PHO,25.0,32.0,32.0,32.0,6.4,14.6,0.440,...,2.7,50.0,13.0,49.0,15.0,52.0,6.0,312.0,23.0,10.0
4,Diana Taurasi,2008,PHO,26.0,34.0,34.0,31.9,7.6,17.0,0.446,...,10.4,44.0,17.0,46.0,20.0,102.0,14.0,276.0,25.0,27.0


In [6]:
for col in player_df.columns:
  print(col)

player
year
tm
age
g
gs
per_game_mp
per_game_fg
per_game_fga
per_game_fg_pct
per_game_3p
per_game_3pa
per_game_3p_pct
per_game_2p
per_game_2pa
per_game_2p_pct
per_game_efg_pct
per_game_ft
per_game_fta
per_game_ft_pct
per_game_orb
per_game_drb
per_game_trb
per_game_ast
per_game_stl
per_game_blk
per_game_tov
per_game_pf
per_game_pts
mp
per_minute_fg
per_minute_fga
per_minute_fg_pct
per_minute_3p
per_minute_3pa
per_minute_3p_pct
per_minute_2p
per_minute_2pa
per_minute_2p_pct
per_minute_ft
per_minute_fta
per_minute_ft_pct
per_minute_orb
per_minute_drb
per_minute_trb
per_minute_ast
per_minute_stl
per_minute_blk
per_minute_tov
per_minute_pf
per_minute_pts
per_poss_fg
per_poss_fga
per_poss_fg_pct
per_poss_3p
per_poss_3pa
per_poss_3p_pct
per_poss_2p
per_poss_2pa
per_poss_2p_pct
per_poss_ft
per_poss_fta
per_poss_ft_pct
per_poss_orb
per_poss_drb
per_poss_trb
per_poss_ast
per_poss_stl
per_poss_blk
per_poss_tov
per_poss_pf
per_poss_pts
per_poss_ortg
per_poss_drtg
advanced_per
advanced_ts_pct
advan

In [10]:
# top 25 per_poss_pts in 2024 with selected columns
player_df[player_df["year"] == 2024] \
    .sort_values("per_poss_pts", ascending=False) \
    .loc[:, ["player", "tm", "per_game_mp", "per_poss_pts", "per_game_pts"]] \
    .head(25)

,player,tm,per_game_mp,per_poss_pts,per_game_pts
729,A'ja Wilson,LVA,34.4,39.2,26.9
762,Chennedy Carter,CHI,26.0,34.4,17.5
807,Kahleah Copper,PHO,32.4,33.4,21.1
746,Breanna Stewart,NYL,32.7,32.0,20.4
750,Brittney Griner,PHO,28.7,31.8,17.8
857,Napheesa Collier,MIN,34.7,30.3,20.4
887,Shakira Austin,WAS,19.8,30.0,11.8
815,Kelsey Mitchell,IND,32.0,30.0,19.2
800,Jewell Loyd,SEA,33.7,29.5,19.7
883,Sabrina Ionescu,NYL,32.1,29.0,18.2


In [11]:
# top 25 per_poss_pts in 2024 with selected columns
player_df[player_df["year"] == 2024] \
    .sort_values("per_game_pts", ascending=False) \
    .loc[:, ["player", "tm", "per_game_mp", "per_game_pts"]] \
    .head(25)

,player,tm,per_game_mp,per_game_pts
729,A'ja Wilson,LVA,34.4,26.9
742,Arike Ogunbowale,DAL,38.6,22.2
807,Kahleah Copper,PHO,32.4,21.1
857,Napheesa Collier,MIN,34.7,20.4
746,Breanna Stewart,NYL,32.7,20.4
800,Jewell Loyd,SEA,33.7,19.7
753,Caitlin Clark,IND,35.4,19.2
815,Kelsey Mitchell,IND,32.0,19.2
883,Sabrina Ionescu,NYL,32.1,18.2
885,Satou Sabally,DAL,34.1,17.9


In [12]:
# filter for 2024 players
df_2024 = player_df[player_df["year"] == 2024]

# full set of 27 personas and their primary sorting stat
personas = {
    # Offensive
    "elite_scorer": "per_game_pts",
    "efficient_scorer": "advanced_ts_pct",
    "volume_shooter": "per_game_fga",
    "three_point_specialist": "per_game_3pa",
    "slasher": "shooting_fg_pct_by_distance_0-3",
    "free_throw_generator": "per_game_fta",
    "and_one_machine": "pbp_misc_and1",

    # Playmaking / IQ
    "playmaker": "per_game_ast",
    "offensive_hub": "advanced_ast_pct",
    "turnover_prone": "per_game_tov",
    "floor_general": "advanced_ast_pct",  # sort by ast_pct, can display tov_pct too
    "plus_minus_driver": "pbp_plus_minus_per_100_poss_on_off",
    "self_creator": "advanced_usg_pct",  # sort by usg_pct, low %astd may be inspected separately

    # Defensive
    "rim_protector": "per_game_blk",
    "steal_artist": "per_game_stl",
    "defensive_anchor": "advanced_dws",
    "glass_cleaner": "per_game_trb",
    "offensive_rebounder": "advanced_orb_pct",
    "defensive_rebounder": "advanced_drb_pct",

    # Shooting types
    "midrange_sniper": "shooting_fg_pct_by_distance_10-16",
    "corner_3_specialist": "shooting_corner_3s_3p_pct",
    "catch_and_shoot": "shooting_pct_of_fg_astd_3p",
    "stretch_big": "per_game_3pa",
    "heave_chucker": "shooting_heaves_att",

    # Misc
    "all_around_star": "advanced_ws",
    "impact_bench": "per_minute_pts",
    "fast_break_threat": "pbp_misc_pga",
}

# collect Top 10 per persona
top_10_personas = {
    label: df_2024.sort_values(stat, ascending=False).loc[:, ["player", "tm", stat]].head(10)
    for label, stat in personas.items()
}

In [13]:
df_2024

,player,year,tm,age,g,gs,per_game_mp,per_game_fg,per_game_fga,per_game_fg_pct,...,pbp_plus_minus_per_100_poss_on_off,pbp_turnovers_badpass,pbp_turnovers_lostball,pbp_fouls_committed_shoot,pbp_fouls_committed_off,pbp_fouls_drawn_shoot,pbp_fouls_drawn_off,pbp_misc_pga,pbp_misc_and1,pbp_misc_blkd
729,A'ja Wilson,2024,LVA,27.0,38.0,38.0,34.4,10.1,19.6,0.518,...,-2.7,20.0,17.0,42.0,9.0,131.0,2.0,223.0,30.0,48.0
730,Aaliyah Edwards,2024,WAS,21.0,34.0,17.0,21.8,3.0,6.2,0.490,...,-7.0,18.0,13.0,42.0,15.0,33.0,10.0,122.0,4.0,23.0
731,Aari McDonald,2024,LAS,25.0,26.0,10.0,21.8,3.0,7.3,0.403,...,3.3,29.0,10.0,14.0,3.0,13.0,13.0,227.0,3.0,10.0
732,Aerial Powers,2024,ATL,30.0,17.0,2.0,17.9,2.9,8.1,0.355,...,-7.3,5.0,8.0,14.0,2.0,14.0,0.0,56.0,3.0,10.0
733,Alanna Smith,2024,MIN,27.0,39.0,39.0,26.5,3.8,8.0,0.471,...,9.7,35.0,20.0,58.0,14.0,32.0,10.0,300.0,9.0,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
907,Tyasha Harris,2024,CON,26.0,39.0,38.0,28.8,3.7,8.7,0.425,...,0.0,39.0,10.0,36.0,1.0,37.0,6.0,264.0,10.0,15.0
908,Veronica Burton,2024,CON,23.0,31.0,1.0,12.7,0.8,2.3,0.361,...,7.7,11.0,3.0,14.0,0.0,12.0,7.0,131.0,1.0,4.0
909,Victaria Saxton,2024,IND,24.0,9.0,0.0,2.6,0.3,1.0,0.333,...,-25.1,0.0,1.0,3.0,0.0,1.0,0.0,0.0,0.0,1.0
910,Victoria Vivians,2024,SEA,29.0,35.0,15.0,12.7,1.2,3.5,0.333,...,1.6,12.0,3.0,24.0,2.0,3.0,3.0,67.0,1.0,8.0


In [14]:
# one-hot encode persona membership: 1 if in top 10 for that persona, else 0
for persona, df_top10 in top_10_personas.items():
    top_players = set(df_top10["player"])
    player_df[persona] = player_df["player"].apply(lambda x: 1 if x in top_players else 0)

In [15]:
player_df

,player,year,tm,age,g,gs,per_game_mp,per_game_fg,per_game_fga,per_game_fg_pct,...,offensive_rebounder,defensive_rebounder,midrange_sniper,corner_3_specialist,catch_and_shoot,stretch_big,heave_chucker,all_around_star,impact_bench,fast_break_threat
0,Diana Taurasi,2004,PHO,22.0,34.0,34.0,33.2,6.1,14.8,0.416,...,0,0,0,0,0,1,0,0,0,0
1,Diana Taurasi,2005,PHO,23.0,33.0,33.0,33.0,5.3,12.9,0.410,...,0,0,0,0,0,1,0,0,0,0
2,Diana Taurasi,2006,PHO,24.0,34.0,34.0,33.9,8.8,19.4,0.452,...,0,0,0,0,0,1,0,0,0,0
3,Diana Taurasi,2007,PHO,25.0,32.0,32.0,32.0,6.4,14.6,0.440,...,0,0,0,0,0,1,0,0,0,0
4,Diana Taurasi,2008,PHO,26.0,34.0,34.0,31.9,7.6,17.0,0.446,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1064,Tyasha Harris,0,Career,26.0,167.0,49.0,19.6,2.4,5.9,0.410,...,0,0,0,0,0,0,0,0,0,0
1065,Veronica Burton,0,Career,23.0,107.0,20.0,14.0,0.7,2.2,0.326,...,0,0,0,0,0,0,0,0,0,0
1066,Victaria Saxton,0,Career,24.0,24.0,0.0,3.2,0.4,1.1,0.333,...,0,0,0,0,0,0,0,0,0,0
1067,Victoria Vivians,0,Career,29.0,179.0,93.0,20.8,2.4,6.8,0.357,...,0,0,0,0,0,0,0,0,0,0


In [16]:
# load the player game logs CSV from the data folder
player_gamelogs_df = pd.read_csv("LHL-final-final-project/data/2024_player_gamelogs.csv")


# preview
player_gamelogs_df.head()

,player,year,month,day,age,tm,home_away,opp,win_margin,gs,...,orb,drb,trb,ast,stl,blk,tov,pf,pts,gmsc
0,Lindsay Allen,2024,5,15,29.2,CHI,2,DAL,-8.0,0,...,0,1,1,0,1,0,0,0,5,3.2
1,Lindsay Allen,2024,5,18,29.2,CHI,2,DAL,9.0,0,...,0,1,1,1,0,0,1,0,2,1.7
2,Lindsay Allen,2024,5,23,29.2,CHI,2,NYL,9.0,0,...,0,1,1,2,0,0,1,3,8,4.2
3,Lindsay Allen,2024,5,25,29.2,CHI,1,CON,-4.0,0,...,1,1,2,2,2,0,2,1,6,7.1
4,Lindsay Allen,2024,5,28,29.2,CHI,1,SEA,-9.0,0,...,0,2,2,4,0,0,3,2,3,0.1


In [17]:
# get list of persona columns
persona_cols = list(top_10_personas.keys())

# subset to player + persona columns
player_personas = player_df[["player"] + persona_cols]

# merge into player_gamelogs_df
player_gamelogs_df = player_gamelogs_df.merge(player_personas, on="player", how="left")

In [20]:
# filter rows for A'ja Wilson
player_gamelogs_df[player_gamelogs_df["player"] == "A'ja Wilson"]

,player,year,month,day,age,tm,home_away,opp,win_margin,gs,...,offensive_rebounder,defensive_rebounder,midrange_sniper,corner_3_specialist,catch_and_shoot,stretch_big,heave_chucker,all_around_star,impact_bench,fast_break_threat
31851,A'ja Wilson,2024,5,14,27.8,LVA,1,PHO,9.0,1,...,0,1,0,0,1,0,1,1,1,0
31852,A'ja Wilson,2024,5,14,27.8,LVA,1,PHO,9.0,1,...,0,1,0,0,1,0,1,1,1,0
31853,A'ja Wilson,2024,5,14,27.8,LVA,1,PHO,9.0,1,...,0,1,0,0,1,0,1,1,1,0
31854,A'ja Wilson,2024,5,14,27.8,LVA,1,PHO,9.0,1,...,0,1,0,0,1,0,1,1,1,0
31855,A'ja Wilson,2024,5,14,27.8,LVA,1,PHO,9.0,1,...,0,1,0,0,1,0,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32150,A'ja Wilson,2024,9,17,28.1,LVA,2,SEA,13.0,1,...,0,1,0,0,1,0,1,1,1,0
32151,A'ja Wilson,2024,9,17,28.1,LVA,2,SEA,13.0,1,...,0,1,0,0,1,0,1,1,1,0
32152,A'ja Wilson,2024,9,17,28.1,LVA,2,SEA,13.0,1,...,0,1,0,0,1,0,1,1,1,0
32153,A'ja Wilson,2024,9,17,28.1,LVA,2,SEA,13.0,1,...,0,1,0,0,1,0,1,1,1,0


In [39]:
# drop duplicate player-game rows before summing personas
player_personas_clean = (
    player_gamelogs_df
    .drop_duplicates(subset=["player", "month", "day", "tm", "opp"])
)

# now group and sum by game
persona_summary = (
    player_personas_clean
    .rename(columns={"tm": "team"})
    .groupby(["month", "day", "team", "opp"])[persona_cols]
    .sum()
    .reset_index()
)

# drop existing persona columns from df before merging
df = df.drop(columns=persona_cols, errors="ignore")

# re-merge into df
df = df.merge(persona_summary, on=["month", "day", "team", "opp"], how="left")

In [40]:
df

,team,g_num,month,day,home_away,opp,win_loss,team_score,opp_score,team_fg,...,offensive_rebounder,defensive_rebounder,midrange_sniper,corner_3_specialist,catch_and_shoot,stretch_big,heave_chucker,all_around_star,impact_bench,fast_break_threat
0,ATL,1,5,15,2,LAS,1,92,81,34,...,1,1,0,0,0,1,2,0,0,0
1,ATL,1,5,15,2,LAS,1,92,81,34,...,1,1,0,0,0,1,2,0,0,0
2,ATL,1,5,15,2,LAS,1,92,81,34,...,1,1,0,0,0,1,2,0,0,0
3,ATL,1,5,15,2,LAS,1,92,81,34,...,1,1,0,0,0,1,2,0,0,0
4,ATL,1,5,15,2,LAS,1,92,81,34,...,1,1,0,0,0,1,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32683,WAS,40,9,19,1,IND,1,92,91,33,...,1,1,2,2,1,0,0,0,0,1
32684,WAS,40,9,19,1,IND,1,92,91,33,...,1,1,2,2,1,0,0,0,0,1
32685,WAS,40,9,19,1,IND,1,92,91,33,...,1,1,2,2,1,0,0,0,0,1
32686,WAS,40,9,19,1,IND,1,92,91,33,...,1,1,2,2,1,0,0,0,0,1


In [52]:
# subset df to just the desired columns
df_personas = df[["team", "opp", "month", "day", "team_score", "opp_score"] + persona_cols]

In [42]:
for col in df_personas.columns:
  print(col)

team
opp
team_score
opp_score
elite_scorer
efficient_scorer
volume_shooter
three_point_specialist
slasher
free_throw_generator
and_one_machine
playmaker
offensive_hub
turnover_prone
floor_general
plus_minus_driver
self_creator
rim_protector
steal_artist
defensive_anchor
glass_cleaner
offensive_rebounder
defensive_rebounder
midrange_sniper
corner_3_specialist
catch_and_shoot
stretch_big
heave_chucker
all_around_star
impact_bench
fast_break_threat


In [43]:
# targets
target_cols = ["team_score", "opp_score"]

# models to train: 12 teams + league-wide
teams = df_personas["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df_personas.copy()
    else:
        df_team = df_personas[df_personas["team"] == team_name].copy()

    # features: drop non-feature columns
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score"])
    y = df_team[["team_score", "opp_score"]]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # track team + opp info from original df
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score", "opp_score"]].reset_index(drop=True)

    # model
    base_model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # predict
    y_pred = model.predict(X_test)

    # compute row-level absolute errors
    team_score_mae = np.abs(y_pred[:, 0] - meta["team_score"].values)
    opp_score_mae = np.abs(y_pred[:, 1] - meta["opp_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i, 0],
            "team_score_mae": team_score_mae[i],
            "opp_score": meta.loc[i, "opp_score"],
            "opp_score_pred": y_pred[i, 1],
            "opp_score_mae": opp_score_mae[i],
        })

    # overall summary metrics
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        results.append({
            "Model": team_name,
            "Target": col,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse
        })

# create final prediction DataFrame
predictions_df = pd.DataFrame(all_predictions)

# preview
predictions_df.head()

,Model,team,opp,team_score,team_score_pred,team_score_mae,opp_score,opp_score_pred,opp_score_mae
0,ATL,ATL,PHO,80,81.092896,1.092896,82,80.239502,1.760498
1,ATL,ATL,CON,78,76.356636,1.643364,74,86.072731,12.072731
2,ATL,ATL,NYL,75,76.356636,1.356636,96,86.072731,9.927269
3,ATL,ATL,IND,79,79.019234,0.019234,91,90.996239,0.003761
4,ATL,ATL,DAL,83,82.440720,0.559280,78,77.419548,0.580452


In [44]:
rows = []

for team, group in predictions_df.groupby('Model'):
    y_team_true = group['team_score']
    y_team_pred = group['team_score_pred']
    y_opp_true = group['opp_score']
    y_opp_pred = group['opp_score_pred']

    rows.append({
        'Model': team,
        'team_score_mae_min': (y_team_true - y_team_pred).abs().min(),
        'team_score_mae_max': (y_team_true - y_team_pred).abs().max(),
        'team_score_mae_mean': (y_team_true - y_team_pred).abs().mean(),
        'team_score_mae_median': (y_team_true - y_team_pred).abs().median(),
        'team_score_r2': r2_score(y_team_true, y_team_pred),
        'opp_score_mae_min': (y_opp_true - y_opp_pred).abs().min(),
        'opp_score_mae_max': (y_opp_true - y_opp_pred).abs().max(),
        'opp_score_mae_mean': (y_opp_true - y_opp_pred).abs().mean(),
        'opp_score_mae_median': (y_opp_true - y_opp_pred).abs().median(),
        'opp_score_r2': r2_score(y_opp_true, y_opp_pred),
    })

model_mae_r2_df = pd.DataFrame(rows)

In [45]:
model_mae_r2_df

,Model,team_score_mae_min,team_score_mae_max,team_score_mae_mean,team_score_mae_median,team_score_r2,opp_score_mae_min,opp_score_mae_max,opp_score_mae_mean,opp_score_mae_median,opp_score_r2
0,ATL,0.000053,25.907104,6.293115,4.440720,0.254019,0.001572,23.760498,5.537890,3.760498,0.302907
1,CHI,0.008904,18.749283,5.663174,3.250717,0.323652,0.001503,14.364807,4.360099,3.351570,0.378879
2,CON,0.062363,19.570190,5.187001,3.385658,0.399602,0.016052,16.888863,4.647594,3.888863,0.636396
3,DAL,0.021431,25.725639,7.272702,5.545822,0.286912,0.005028,19.176544,5.796204,4.949066,0.355700
4,IND,0.010536,21.897430,6.879140,5.584373,0.279322,0.031509,19.322968,6.557067,4.212883,0.155525
5,LAS,0.012444,31.041130,7.722032,5.588615,0.096094,0.051926,33.791451,6.320523,4.791451,0.160090
6,LVA,0.000237,21.423622,5.704933,2.576378,0.164195,0.000954,19.321121,7.111643,4.678879,0.097530
7,League,0.001030,31.998970,6.910147,5.513908,0.320997,0.008461,34.603317,6.706646,5.157112,0.364293
8,MIN,0.002304,20.377838,6.853542,6.312904,0.392354,0.000793,19.391891,6.785696,6.608109,0.123050
9,NYL,0.003731,24.394836,5.592291,4.274277,0.373474,0.034958,16.244270,5.552386,4.641930,0.282398


In [47]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract and format for team_score
importances_team = model.estimators_[0].feature_importances_
importances_opp = model.estimators_[1].feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team,
    "Importance_opp_score": importances_opp
})

# sort if desired
df_importances = df_importances.sort_values("Importance_team_score", ascending=False)

In [48]:
df_importances

,Feature,Importance_team_score,Importance_opp_score
0,elite_scorer,0.144599,0.027743
16,glass_cleaner,0.075133,0.015995
19,midrange_sniper,0.073075,0.016130
25,impact_bench,0.070392,0.027439
15,defensive_anchor,0.063807,0.305623
20,corner_3_specialist,0.048902,0.012644
23,heave_chucker,0.047701,0.020282
18,defensive_rebounder,0.044705,0.023092
8,offensive_hub,0.043300,0.028052
24,all_around_star,0.042750,0.013018


In [53]:
# define final columns
df_personas_v2 = df[["team", "opp", "month", "day", "team_score", "opp_score", "advanced_pace"] + persona_cols].copy()

In [63]:
# targets
target_cols = ["team_score", "opp_score"]

# models to train: 12 teams + league-wide
teams = df_personas_v2["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df_personas_v2.copy()
    else:
        df_team = df_personas_v2[df_personas_v2["team"] == team_name].copy()

    # features: drop non-feature columns
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team[["team_score", "opp_score"]]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # track team + opp info from original df
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score", "opp_score"]].reset_index(drop=True)

    # model
    base_model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # predict
    y_pred = model.predict(X_test)

    # compute row-level absolute errors
    team_score_mae = np.abs(y_pred[:, 0] - meta["team_score"].values)
    opp_score_mae = np.abs(y_pred[:, 1] - meta["opp_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i, 0],
            "team_score_mae": team_score_mae[i],
            "opp_score": meta.loc[i, "opp_score"],
            "opp_score_pred": y_pred[i, 1],
            "opp_score_mae": opp_score_mae[i],
        })

    # overall summary metrics
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        results.append({
            "Model": team_name,
            "Target": col,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse
        })

# create final prediction DataFrame
predictions_df = pd.DataFrame(all_predictions)

# preview
predictions_df.head()

,Model,team,opp,team_score,team_score_pred,team_score_mae,opp_score,opp_score_pred,opp_score_mae
0,ATL,ATL,PHO,80,80.958778,0.958778,82,81.897278,0.102722
1,ATL,ATL,CON,78,78.161003,0.161003,74,74.491669,0.491669
2,ATL,ATL,NYL,75,75.360474,0.360474,96,95.677528,0.322472
3,ATL,ATL,IND,79,78.408951,0.591049,91,90.606651,0.393349
4,ATL,ATL,DAL,83,83.013870,0.013870,78,77.804314,0.195686


In [64]:
rows = []

for team, group in predictions_df.groupby('Model'):
    y_team_true = group['team_score']
    y_team_pred = group['team_score_pred']
    y_opp_true = group['opp_score']
    y_opp_pred = group['opp_score_pred']

    rows.append({
        'Model': team,
        'team_score_mae_min': (y_team_true - y_team_pred).abs().min(),
        'team_score_mae_max': (y_team_true - y_team_pred).abs().max(),
        'team_score_mae_mean': (y_team_true - y_team_pred).abs().mean(),
        'team_score_mae_median': (y_team_true - y_team_pred).abs().median(),
        'team_score_r2': r2_score(y_team_true, y_team_pred),
        'opp_score_mae_min': (y_opp_true - y_opp_pred).abs().min(),
        'opp_score_mae_max': (y_opp_true - y_opp_pred).abs().max(),
        'opp_score_mae_mean': (y_opp_true - y_opp_pred).abs().mean(),
        'opp_score_mae_median': (y_opp_true - y_opp_pred).abs().median(),
        'opp_score_r2': r2_score(y_opp_true, y_opp_pred),
    })

model_mae_r2_df = pd.DataFrame(rows)

model_mae_r2_df

,Model,team_score_mae_min,team_score_mae_max,team_score_mae_mean,team_score_mae_median,team_score_r2,opp_score_mae_min,opp_score_mae_max,opp_score_mae_mean,opp_score_mae_median,opp_score_r2
0,ATL,0.000504,1.846825,0.479529,0.380173,0.996571,0.007469,8.704483,0.840089,0.393349,0.952151
1,CHI,0.010849,14.264511,0.827884,0.288971,0.936087,0.008293,9.304390,0.541916,0.234337,0.954584
2,CON,0.040482,8.712906,1.507277,0.788071,0.934260,0.047760,16.049026,1.531922,0.423271,0.870789
3,DAL,0.003830,3.599144,0.750895,0.437965,0.990253,0.028526,5.424835,0.667685,0.306892,0.983740
4,IND,0.012070,4.719566,1.036104,0.623398,0.977498,0.035568,8.296684,0.946579,0.566139,0.953070
5,LAS,0.045456,7.785027,1.135994,0.578575,0.965541,0.003372,10.444977,0.846982,0.349411,0.950371
6,LVA,0.038689,10.519814,1.107334,0.614120,0.930248,0.001534,15.851616,1.376794,0.599014,0.869843
7,League,0.013245,21.431770,5.256214,4.292099,0.610995,0.003593,18.957115,5.144882,4.109444,0.633839
8,MIN,0.004707,1.963097,0.628030,0.490494,0.994886,0.038292,2.824127,1.046425,0.778641,0.980189
9,NYL,0.026649,1.765549,0.372942,0.253555,0.997679,0.007286,6.091698,1.009194,0.629181,0.963485


In [65]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract and format for team_score
importances_team = model.estimators_[0].feature_importances_
importances_opp = model.estimators_[1].feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team,
    "Importance_opp_score": importances_opp
})

# sort if desired
df_importances = df_importances.sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score,Importance_opp_score
27,fast_break_threat,0.173234,0.015467
3,volume_shooter,0.100797,0.120629
1,elite_scorer,0.061273,0.015299
25,all_around_star,0.047335,0.246934
20,midrange_sniper,0.047073,0.011951
16,defensive_anchor,0.045361,0.148724
26,impact_bench,0.041224,0.019351
17,glass_cleaner,0.039021,0.011443
19,defensive_rebounder,0.038834,0.036594
9,offensive_hub,0.037867,0.010788


In [66]:
predictions_df.query("Model != 'League'")[["team_score_mae", "opp_score_mae"]].describe()

,team_score_mae,opp_score_mae
count,8178.000000,8178.000000
mean,0.840287,0.910204
std,1.451925,1.921525
min,0.000504,0.001534
25%,0.217049,0.189743
50%,0.461937,0.453964
75%,0.887177,0.817574
max,14.264511,16.049026


In [67]:
# features and targets
X = df_personas_v2.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = df_personas_v2[["team_score", "opp_score"]]

# cross-validation setup
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# store fold scores
cv_results = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = MultiOutputRegressor(XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    ))

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # compute MAE and R² for both targets
    for i, col in enumerate(["team_score", "opp_score"]):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])

        cv_results.append({
            "Fold": fold + 1,
            "Target": col,
            "MAE": mae,
            "R2": r2
        })

# convert to DataFrame
cv_results_df = pd.DataFrame(cv_results)

# view
cv_results_df.groupby("Target").agg({"MAE": ["mean", "std"], "R2": ["mean", "std"]})

MAE                  R2          
                mean       std      mean       std
Target                                            
opp_score   5.134101  0.061119  0.639193  0.010689
team_score  5.298341  0.055306  0.606463  0.009904

In [68]:
# list of teams
teams = df_personas_v2["team"].unique()

# store team-level CV results
team_cv_results = []

for team_name in teams:
    df_team = df_personas_v2[df_personas_v2["team"] == team_name].copy()

    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team[["team_score", "opp_score"]]

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model = MultiOutputRegressor(XGBRegressor(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=1,
            min_child_weight=1,
            random_state=42,
            verbosity=0
        ))

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        for i, col in enumerate(["team_score", "opp_score"]):
            mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
            r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])

            team_cv_results.append({
                "Team": team_name,
                "Fold": fold + 1,
                "Target": col,
                "MAE": mae,
                "R2": r2
            })

# compile and summarize
team_cv_df = pd.DataFrame(team_cv_results)

# mean/std per team per target
team_cv_summary = team_cv_df.groupby(["Team", "Target"]).agg(
    MAE_mean=("MAE", "mean"),
    MAE_std=("MAE", "std"),
    R2_mean=("R2", "mean"),
    R2_std=("R2", "std")
).reset_index()

# view
team_cv_summary.head()

,Team,Target,MAE_mean,MAE_std,R2_mean,R2_std
0,ATL,opp_score,0.866606,0.078695,0.948566,0.008922
1,ATL,team_score,0.477690,0.006476,0.996531,0.000295
2,CHI,opp_score,0.641393,0.078315,0.940575,0.011433
3,CHI,team_score,1.050375,0.139136,0.916216,0.015321
4,CON,opp_score,1.535865,0.191257,0.869675,0.030487


In [69]:
df_team = df_personas_v2[df_personas_v2["team"] == "ATL"].copy()

# keep inputs the same
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# shuffle the targets
y_shuffled = df_team[["team_score", "opp_score"]].sample(frac=1, random_state=42).reset_index(drop=True)

In [70]:
# choose one team
team_name = "ATL"
df_team = df_personas_v2[df_personas_v2["team"] == team_name].copy()

# features
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# shuffle the targets
y = df_team[["team_score", "opp_score"]].sample(frac=1, random_state=42).reset_index(drop=True)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# model
model = MultiOutputRegressor(XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=1,
    min_child_weight=1,
    random_state=42,
    verbosity=0
))
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# evaluate
for i, col in enumerate(["team_score", "opp_score"]):
    mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
    r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
    print(f"[{team_name}] {col} — MAE: {mae:.3f}, R²: {r2:.3f}")

[ATL] team_score — MAE: 8.024, R²: -0.013
[ATL] opp_score — MAE: 7.252, R²: -0.005


In [72]:
# preview predictions vs shuffled targets
pred_df = pd.DataFrame({
    "team_score_true": y_test["team_score"].values,
    "team_score_pred": y_pred[:, 0],
    "opp_score_true": y_test["opp_score"].values,
    "opp_score_pred": y_pred[:, 1],
})

# show top rows
pred_df.head(10)

,team_score_true,team_score_pred,opp_score_true,opp_score_pred
0,55,76.833199,68,79.167397
1,77,76.002876,85,80.009003
2,79,76.293533,84,79.594353
3,92,79.454376,81,80.868408
4,86,76.115784,70,77.456734
5,83,78.247025,81,81.696739
6,70,77.417046,81,80.460426
7,66,76.995865,74,79.086906
8,100,77.093712,104,79.438858
9,107,74.401260,96,78.272430


In [73]:
# targets
target_cols = ["team_score", "opp_score"]

# models to train: 12 teams + league-wide
teams = df_personas["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df_personas.copy()
    else:
        df_team = df_personas[df_personas["team"] == team_name].copy()

    # features: drop non-feature columns
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team[["team_score", "opp_score"]]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # track team + opp info from original df
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score", "opp_score"]].reset_index(drop=True)

    # model
    base_model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # predict
    y_pred = model.predict(X_test)

    # compute row-level absolute errors
    team_score_mae = np.abs(y_pred[:, 0] - meta["team_score"].values)
    opp_score_mae = np.abs(y_pred[:, 1] - meta["opp_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i, 0],
            "team_score_mae": team_score_mae[i],
            "opp_score": meta.loc[i, "opp_score"],
            "opp_score_pred": y_pred[i, 1],
            "opp_score_mae": opp_score_mae[i],
        })

    # overall summary metrics
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        results.append({
            "Model": team_name,
            "Target": col,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse
        })

# create final prediction DataFrame
predictions_df = pd.DataFrame(all_predictions)

# preview
predictions_df.head()

,Model,team,opp,team_score,team_score_pred,team_score_mae,opp_score,opp_score_pred,opp_score_mae
0,ATL,ATL,PHO,80,81.132828,1.132828,82,80.228828,1.771172
1,ATL,ATL,CON,78,76.372932,1.627068,74,85.951202,11.951202
2,ATL,ATL,NYL,75,76.372932,1.372932,96,85.951202,10.048798
3,ATL,ATL,IND,79,78.976120,0.023880,91,90.979187,0.020813
4,ATL,ATL,DAL,83,82.398903,0.601097,78,77.370651,0.629349


In [74]:
rows = []

for team, group in predictions_df.groupby('Model'):
    y_team_true = group['team_score']
    y_team_pred = group['team_score_pred']
    y_opp_true = group['opp_score']
    y_opp_pred = group['opp_score_pred']

    rows.append({
        'Model': team,
        'team_score_mae_min': (y_team_true - y_team_pred).abs().min(),
        'team_score_mae_max': (y_team_true - y_team_pred).abs().max(),
        'team_score_mae_mean': (y_team_true - y_team_pred).abs().mean(),
        'team_score_mae_median': (y_team_true - y_team_pred).abs().median(),
        'team_score_r2': r2_score(y_team_true, y_team_pred),
        'opp_score_mae_min': (y_opp_true - y_opp_pred).abs().min(),
        'opp_score_mae_max': (y_opp_true - y_opp_pred).abs().max(),
        'opp_score_mae_mean': (y_opp_true - y_opp_pred).abs().mean(),
        'opp_score_mae_median': (y_opp_true - y_opp_pred).abs().median(),
        'opp_score_r2': r2_score(y_opp_true, y_opp_pred),
    })

model_mae_r2_df = pd.DataFrame(rows)

model_mae_r2_df

,Model,team_score_mae_min,team_score_mae_max,team_score_mae_mean,team_score_mae_median,team_score_r2,opp_score_mae_min,opp_score_mae_max,opp_score_mae_mean,opp_score_mae_median,opp_score_r2
0,ATL,0.003067,25.867172,6.205669,4.398903,0.258934,0.015656,23.771172,5.450927,3.370651,0.301666
1,CHI,0.000656,18.832985,5.557410,3.211021,0.366355,0.008835,14.131195,4.428608,3.803772,0.354322
2,CON,0.083122,19.409950,5.354064,3.336197,0.359178,0.006157,16.598618,4.712160,3.598618,0.620504
3,DAL,0.039459,25.880974,7.082463,5.226807,0.323773,0.017815,19.133179,5.750525,4.865982,0.359402
4,IND,0.002113,21.874573,6.823081,5.672226,0.285640,0.001770,19.201683,6.522211,4.094643,0.179244
5,LAS,0.015373,30.890282,7.797033,5.646423,0.092992,0.028511,33.671013,6.212072,4.671013,0.162031
6,LVA,0.000443,21.480835,5.724631,2.519165,0.161792,0.002258,19.118095,7.060314,4.881905,0.108337
7,League,0.045952,32.132690,6.939911,5.640869,0.321013,0.014000,34.501610,6.675929,5.106255,0.373810
8,MIN,0.000031,20.583618,6.998748,6.383774,0.353795,0.002357,19.308006,6.912356,6.691994,0.128361
9,NYL,0.013199,24.311707,5.584274,3.729279,0.392737,0.021759,16.124344,5.571694,4.686920,0.283691


In [ ]:
# load the player game logs CSV from the data folder
df = pd.read_csv("LHL-final-final-project/data/2024_merged_gamelogs.csv")


# preview
df.head()

In [76]:
team_pace_df = (
    df.groupby("team")["advanced_pace"]
    .mean()
    .rename("team_avg_advanced_pace")
    .reset_index()
)

In [77]:
team_pace_df

,team,team_avg_advanced_pace
0,ATL,77.224267
1,CHI,78.009222
2,CON,75.675688
3,DAL,80.206247
4,IND,79.831345
5,LAS,79.147529
6,LVA,79.552446
7,MIN,77.596794
8,NYL,78.038919
9,PHO,78.214742


In [78]:
# 2. Drop the game-level advanced_pace from df_personas_v2 (if it exists)
df_personas_v2 = df_personas_v2.drop(columns=["advanced_pace"], errors="ignore")

# 3. Merge in team average pace
df_personas_v2 = df_personas_v2.merge(team_pace_df, on="team", how="left")

In [79]:
# targets
target_cols = ["team_score", "opp_score"]

# models to train: 12 teams + league-wide
teams = df_personas_v2["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df_personas_v2.copy()
    else:
        df_team = df_personas_v2[df_personas_v2["team"] == team_name].copy()

    # features: drop non-feature columns
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team[["team_score", "opp_score"]]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # track team + opp info from original df
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score", "opp_score"]].reset_index(drop=True)

    # model
    base_model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # predict
    y_pred = model.predict(X_test)

    # compute row-level absolute errors
    team_score_mae = np.abs(y_pred[:, 0] - meta["team_score"].values)
    opp_score_mae = np.abs(y_pred[:, 1] - meta["opp_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i, 0],
            "team_score_mae": team_score_mae[i],
            "opp_score": meta.loc[i, "opp_score"],
            "opp_score_pred": y_pred[i, 1],
            "opp_score_mae": opp_score_mae[i],
        })

    # overall summary metrics
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        results.append({
            "Model": team_name,
            "Target": col,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse
        })

# create final prediction DataFrame
predictions_df = pd.DataFrame(all_predictions)

# preview
predictions_df.head()

,Model,team,opp,team_score,team_score_pred,team_score_mae,opp_score,opp_score_pred,opp_score_mae
0,ATL,ATL,PHO,80,81.132828,1.132828,82,80.228828,1.771172
1,ATL,ATL,CON,78,76.372932,1.627068,74,85.951202,11.951202
2,ATL,ATL,NYL,75,76.372932,1.372932,96,85.951202,10.048798
3,ATL,ATL,IND,79,78.976120,0.023880,91,90.979187,0.020813
4,ATL,ATL,DAL,83,82.398903,0.601097,78,77.370651,0.629349


In [80]:
rows = []

for team, group in predictions_df.groupby('Model'):
    y_team_true = group['team_score']
    y_team_pred = group['team_score_pred']
    y_opp_true = group['opp_score']
    y_opp_pred = group['opp_score_pred']

    rows.append({
        'Model': team,
        'team_score_mae_min': (y_team_true - y_team_pred).abs().min(),
        'team_score_mae_max': (y_team_true - y_team_pred).abs().max(),
        'team_score_mae_mean': (y_team_true - y_team_pred).abs().mean(),
        'team_score_mae_median': (y_team_true - y_team_pred).abs().median(),
        'team_score_r2': r2_score(y_team_true, y_team_pred),
        'opp_score_mae_min': (y_opp_true - y_opp_pred).abs().min(),
        'opp_score_mae_max': (y_opp_true - y_opp_pred).abs().max(),
        'opp_score_mae_mean': (y_opp_true - y_opp_pred).abs().mean(),
        'opp_score_mae_median': (y_opp_true - y_opp_pred).abs().median(),
        'opp_score_r2': r2_score(y_opp_true, y_opp_pred),
    })

model_mae_r2_df = pd.DataFrame(rows)

model_mae_r2_df

,Model,team_score_mae_min,team_score_mae_max,team_score_mae_mean,team_score_mae_median,team_score_r2,opp_score_mae_min,opp_score_mae_max,opp_score_mae_mean,opp_score_mae_median,opp_score_r2
0,ATL,0.003067,25.867172,6.205669,4.398903,0.258934,0.015656,23.771172,5.450927,3.370651,0.301666
1,CHI,0.000656,18.832985,5.557410,3.211021,0.366355,0.008835,14.131195,4.428608,3.803772,0.354322
2,CON,0.083122,19.409950,5.354064,3.336197,0.359178,0.006157,16.598618,4.712160,3.598618,0.620504
3,DAL,0.039459,25.880974,7.082463,5.226807,0.323773,0.017815,19.133179,5.750525,4.865982,0.359402
4,IND,0.002113,21.874573,6.823081,5.672226,0.285640,0.001770,19.201683,6.522211,4.094643,0.179244
5,LAS,0.015373,30.890282,7.797033,5.646423,0.092992,0.028511,33.671013,6.212072,4.671013,0.162031
6,LVA,0.000443,21.480835,5.724631,2.519165,0.161792,0.002258,19.118095,7.060314,4.881905,0.108337
7,League,0.007484,32.164261,6.954638,5.665955,0.317450,0.003601,34.750381,6.670267,5.220062,0.375265
8,MIN,0.000031,20.583618,6.998748,6.383774,0.353795,0.002357,19.308006,6.912356,6.691994,0.128361
9,NYL,0.013199,24.311707,5.584274,3.729279,0.392737,0.021759,16.124344,5.571694,4.686920,0.283691


In [81]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract and format for team_score
importances_team = model.estimators_[0].feature_importances_
importances_opp = model.estimators_[1].feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team,
    "Importance_opp_score": importances_opp
})

# sort if desired
df_importances = df_importances.sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score,Importance_opp_score
0,elite_scorer,0.118369,0.014447
16,glass_cleaner,0.093243,0.011944
25,impact_bench,0.080150,0.023794
19,midrange_sniper,0.075198,0.016046
27,team_avg_advanced_pace,0.066542,0.117072
23,heave_chucker,0.059302,0.023697
20,corner_3_specialist,0.051565,0.013588
15,defensive_anchor,0.048239,0.334382
8,offensive_hub,0.040520,0.011133
2,volume_shooter,0.034179,0.047827
